# ViT benchmark experiments

Frozen OpenAI CLIP ViT-L/14 with a supervised linear classifier. The corrected `benchmark` uses matching JPEG settings and fixed split identities. The completed ADM pilot remains separate.

This notebook and the Slurm runner execute the same small modules in `code/vit/engine/`. Read `docs/benchmark.md` for validation, GPU smoke tests and progress commands.


## 1. Settings

Settings live in `code/vit/engine/settings.py`. Defaults: eight holdouts, k=1/2/4/7, three dataset repeats, seed 17, baseline and augmented: 192 runs. Train/validation/test contain 1400/280/280 images per class. Batch jobs select one holdout each and use separate output folders.


In [ ]:
from pathlib import Path
import os
_candidates = [Path(globals().get('__file__', '.')).resolve().parent, Path.cwd(), *Path.cwd().parents]
VIT_ENGINE_ROOT = next(p / 'code/vit/engine' for p in _candidates if (p / 'code/vit/engine').is_dir())
exec(compile((VIT_ENGINE_ROOT / 'settings.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'settings.py'), 'exec'))


## 2. Environment and GPU check

Use the existing `code/vit/setup_carc_vit.sh` once if packages are missing. This notebook never
installs packages into a running kernel. A real GPU operation is checked below.

In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'imports.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'imports.py'), 'exec'))


## 3. Utilities, protocol validation, and run identities

Each experiment/mode/seed has one stable run directory. Matching completed runs are
skipped; configuration or code changes are rejected instead of overwriting results.
Only one process may write a run at a time. If a CARC job is killed, Section 8 explains
how to remove its stale lock after confirming the old job is no longer running.

In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'utilities.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'utilities.py'), 'exec'))


## 4. Read and verify the dataset team's CSVs

Verify class counts, generator identities, path separation, SHA-256 separation, and
image availability before training. By default, verify actual file hashes as well.
Paths from Colab are remapped to the configured CARC dataset root. No new splits are made.
Exact hashes do not detect near-duplicate images; the dataset team owns that audit.

In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'manifests.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'manifests.py'), 'exec'))


## 5. Frozen CLIP, augmentation, and deterministic feature caching

The baseline preserves the starter's CLIP preprocessing and L2-normalized features.
Augmentation happens after CLIP's resize/center-crop and before tensor conversion and
normalization. The encoder remains frozen and in evaluation mode in both conditions.
Augmented training encodes new image views every epoch. Only deterministic baseline,
validation, and evaluation features are cached. Caches are keyed by ordered image
hashes, model/preprocessing, implementation, package versions, and degradation settings.

In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'features.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'features.py'), 'exec'))


## 6. Common evaluation metrics and calibration

Real = 0; fake = 1. F1/precision/recall refer to fake. AUROC uses probabilities.
Calibration uses equal-width bins of **fake probability**, including p=0 and p=1;
ECE weights each bin's absolute probability/fake-frequency gap by its sample count.
Brier score and binary log loss are also reported. No thresholds or temperatures are
fitted on test data. Generator/family tables pair fake images with the same real pool;
metrics such as AUROC/F1 are never computed on fake-only groups.

In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'metrics.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'metrics.py'), 'exec'))


## 7. Train one fresh head, checkpoint each epoch, and evaluate

Baseline training matches the starter's AdamW/BCE setup. Augmented image features are
generated every epoch, then the head uses the same head minibatch size as baseline.
The backbone never receives gradients. A completed epoch is an atomic resume point:
if interrupted mid-epoch, that epoch is repeated with the same deterministic seed.
The best epoch uses validation AUROC, then validation BCE as a tie-breaker.

In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'training_helpers.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'training_helpers.py'), 'exec'))
exec(compile((VIT_ENGINE_ROOT / 'training.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'training.py'), 'exec'))


## 8. Validate the benchmark and selected experiments

Require the installed-image acceptance audit, validate release fingerprints and CSV hashes, then display the selected plan.


In [ ]:
exec(compile((VIT_ENGINE_ROOT / 'preflight.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'preflight.py'), 'exec'))


## 9. Train, resume and evaluate

Run one writer per output folder. Both modes use fresh checkpoints for this corrected benchmark. Test results never select checkpoints or thresholds.


In [ ]:
session_rows = []
exec(compile((VIT_ENGINE_ROOT / 'experiments.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'experiments.py'), 'exec'))


## 10. Collect results, paired augmentation comparisons, and figures

Only the currently selected plan and evaluation settings are collected. Missing results
are listed. Means/std across repeats describe variation in generator choice and sampled
images, not independent confidence intervals. Additional training seeds are separate.
Test images can differ between repeats/k values; augmentation comparisons are paired
within the same experiment and seed. Family summaries average holdout-level results;
they are descriptive and require actual GAN/diffusion holdouts for a family comparison.

In [ ]:
rows, missing = [], []
exec(compile((VIT_ENGINE_ROOT / 'summary.py').read_text(encoding='utf-8'), str(VIT_ENGINE_ROOT / 'summary.py'), 'exec'))


## Interpretation

Tests stay fixed across k/repeats within each holdout. Real split identities are shared. JPEG quantization tables and chroma sampling match; historical compression, image dimensions and semantic distributions can still differ. Baseline and augmented are separate treatments; no domain-adversarial training or zero-shot comparison is implemented. Dataset repeats and a single classifier seed are not independent test-image samples.
